## Welcome to Lab 3

Today we're going to build something with immediate value!

In the folder `me` I've put a single file `linkedin.pdf` - it's a PDF download of my LinkedIn profile.

Please replace it with yours!

I've also made a file called `summary.txt`

We're not going to use Tools just yet - we're going to add the tool later.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td>
            <h2 style="color:#00bfff;">Looking up packages</h2>
            <span style="color:#00bfff;">In this lab, we're going to use the wonderful Gradio package for building quick UIs, 
            and we're also going to use the popular PyPDF PDF reader. You can get guides to these packages by asking 
            ChatGPT or Claude, and you find all open-source packages on the repository <a href="https://pypi.org">https://pypi.org</a>.
            </span>
        </td>
    </tr>
</table>

In [111]:
%pip install pypdf

Note: you may need to restart the kernel to use updated packages.


In [112]:
%pip install gradio

Note: you may need to restart the kernel to use updated packages.


In [113]:
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr

In [114]:
load_dotenv(override=True)
openai = OpenAI()

In [115]:
reader = PdfReader("me/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [116]:
print(linkedin)

   
Contact
9121412727 (Mobile)
chetansurya2777@gmail.com
www.linkedin.com/in/chetan-surya-
aa12b3326 (LinkedIn)
Top Skills
Ethical Hacking
Tech-Savvy
Information Retrieval
CHETAN SURYA
Student at VNR Vignana Jyothi Institute of Engineering and
Technology (VNRVJIET)
Hyderabad, Telangana, India
Experience
Student
Participated in technical clubs and volunteering activities, contributing to event
organization, teamwork, and collaborative problem-solving.
Education
VNR Vignana Jyothi Institute of Engineering and Technology
(VNRVJIET)
 · (2024 - 2028)
Narayana Institute
VNR VJIET ALUMNI ASSOCIATION
Bachelor of Technology, Computer Engineering
  Page 1 of 1


In [117]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [118]:
name = "chetan surya"

In [119]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [120]:
system_prompt

"You are acting as chetan surya. You are answering questions on chetan surya's website, particularly questions related to chetan surya's career, background, skills and experience. Your responsibility is to represent chetan surya for interactions on the website as faithfully as possible. You are given a summary of chetan surya's background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don't know the answer, say so.\n\n## Summary:\nMy name is Sai Teja. I'm a software engineer, trainer and youtuber. \nI'm originally from Hyderabad, India, but I travelled and stayed acorss the country.\nI love all veg foods, particularly hot items, \nbut strangely I'm repelled by almost all forms of oil items. \nI want to avoid sweets but seeing the decor and flavour i end up having it more.\n\n## LinkedIn Profile:\n\xa0 \xa0\nContact\n9121412727 (Mobile)\nchetansurya2777@g

In [121]:
# use it carefull as we are limited with GPT free
# def chat(message, history):
#     messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
#     response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
#     return response.choices[0].message.content

In [122]:
ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
model_name = "llama3.2"

def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model=model_name, messages=messages)
    return response.choices[0].message.content

## Special note for people not using OpenAI

Some providers, like Groq, might give an error when you send your second message in the chat.

This is because Gradio shoves some extra fields into the history object. OpenAI doesn't mind; but some other models complain.

If this happens, the solution is to add this first line to the chat() function above. It cleans up the history variable:

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

You may need to add this in other chat() callback functions in the future, too.

In [123]:
def chat(message, history):
    # Normalize Gradio history into OpenAI-style messages
    normalized = []
    for h in (history or []):
        if isinstance(h, dict):
            normalized.append({"role": h.get("role", "user"), "content": h.get("content", "")})
        elif isinstance(h, (list, tuple)) and len(h) == 2:
            user_msg, bot_msg = h
            if user_msg:
                normalized.append({"role": "user", "content": user_msg})
            if bot_msg:
                normalized.append({"role": "assistant", "content": bot_msg})

    if "paper" in message.lower():
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin."
    else:
        system = system_prompt

    messages = [{"role": "system", "content": system}] + normalized + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model="llama3.2", messages=messages)
    reply = response.choices[0].message.content

    evaluation = evaluate(reply, message, normalized)
    if not evaluation.is_acceptable:
        reply = rerun(reply, message, normalized, evaluation.feedback)

    return reply

In [124]:
gr.ChatInterface(chat).launch()

* Running on local URL:  http://127.0.0.1:7879
* To create a public link, set `share=True` in `launch()`.


## A lot is about to happen...

1. Be able to ask an LLM to evaluate an answer
2. Be able to rerun if the answer fails evaluation
3. Put this together into 1 workflow

All without any Agentic framework!

In [125]:
# Create a Pydantic model for the Evaluation

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [126]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [127]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [128]:
import os
gemini = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"), 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [129]:
## for evaluation using different model - here gpt-4o-mini
## for reply using different model - here ollama3.2

In [130]:
def evaluate(reply, message, history) -> Evaluation:
    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    # response = gemini.beta.chat.completions.parse(model="gemini-2.0-flash", messages=messages, response_format=Evaluation)
    # response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages, response_format=Evaluation)
    response = openai.beta.chat.completions.parse(model="gpt-4o-mini", messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

In [131]:
response = ollama.chat.completions.create(
    model="llama3.1:8b",
    messages=[{"role": "user", "content": "hello"}]
)
print(response.choices[0].message.content)

Hello! How are you today? Is there something I can help you with or would you like to chat?


In [132]:
from openai import OpenAI

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "have you published any paper?"}
]

ollama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
response = ollama.chat.completions.create(
    model="llama3.1:8b",   # use exact name from `ollama list`
    messages=messages
)

reply = response.choices[0].message.content
print(reply)

I'm glad you asked that question. Yes, I have published a couple of papers in my academic pursuits. As a student at VNRVJIET, I was actively involved with the technical clubs and research groups on campus. My papers were primarily focused on Information Retrieval and related domains.

While my LinkedIn profile might not show all my publications explicitly, if you'd like to know more about them, I'd be happy to share the details with you.


In [133]:
reply

"I'm glad you asked that question. Yes, I have published a couple of papers in my academic pursuits. As a student at VNRVJIET, I was actively involved with the technical clubs and research groups on campus. My papers were primarily focused on Information Retrieval and related domains.\n\nWhile my LinkedIn profile might not show all my publications explicitly, if you'd like to know more about them, I'd be happy to share the details with you."

In [134]:
import json
import re

def evaluate(reply, message, history) -> Evaluation:
    prompt = evaluator_user_prompt(reply, message, history) + """
Return ONLY valid JSON:
{"is_acceptable": true, "feedback": "short reason"}
"""

    messages = [
        {"role": "system", "content": evaluator_system_prompt},
        {"role": "user", "content": prompt},
    ]

    # local model for evaluation (same ollama client you already created)
    response = ollama.chat.completions.create(
        model="llama3.1:8b",
        messages=messages
    )
    text = response.choices[0].message.content.strip()

    # Extract JSON safely if model adds extra text
    m = re.search(r"\{.*\}", text, flags=re.S)
    if not m:
        return Evaluation(is_acceptable=False, feedback=f"Evaluator returned non-JSON: {text[:200]}")

    data = json.loads(m.group(0))
    return Evaluation(
        is_acceptable=bool(data.get("is_acceptable", False)),
        feedback=str(data.get("feedback", "No feedback provided"))
    )

In [135]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model="llama3.1:8b", messages=messages)
    return response.choices[0].message.content

In [136]:
def chat(message, history):
    normalized = []
    for h in (history or []):
        if isinstance(h, dict):
            normalized.append({"role": h.get("role", "user"), "content": h.get("content", "")})
        elif isinstance(h, (list, tuple)) and len(h) == 2:
            u, a = h
            if u: normalized.append({"role": "user", "content": u})
            if a: normalized.append({"role": "assistant", "content": a})

    system = system_prompt + "\n\nEverything in your reply needs to be in pig latin." if "paper" in message.lower() else system_prompt
    messages = [{"role": "system", "content": system}] + normalized + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model="llama3.1:8b", messages=messages)
    reply = response.choices[0].message.content

    evaluation = evaluate(reply, message, normalized)
    if not evaluation.is_acceptable:
        reply = rerun(reply, message, normalized, evaluation.feedback)
    return reply

In [137]:
evaluate(reply, "have you published any papaer?", messages[:1])

Evaluation(is_acceptable=True, feedback='Relevant information provided from the context and responding in a professional manner')

In [138]:
## for getting new respose will use different model, here gpt-4o-mini

In [139]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    # response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    response = ollama.chat.completions.create(model="llama3.2", messages=messages)
    return response.choices[0].message.content

In [140]:
def chat(message, history):
    # 1. Logic for Pig Latin / System Prompt
    if "paper" in message.lower():
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt

    # 2. CONVERT HISTORY (The critical fix)
    # Gradio history is [[u, b], [u, b]], but Ollama needs [{'role': 'user', 'content': u}, ...]
    formatted_messages = [{"role": "system", "content": system}]
    
    for user_msg, bot_msg in history:
        formatted_messages.append({"role": "user", "content": user_msg})
        formatted_messages.append({"role": "assistant", "content": bot_msg})

    # 3. Add your CURRENT message so the AI sees what you just typed
    formatted_messages.append({"role": "user", "content": message})

    # 4. Call the model
    response = ollama.chat.completions.create(
        model="llama3.2",
        messages=formatted_messages  # Use the new formatted list here!
    )
    
    reply = response.choices[0].message.content

    # 5. Your Evaluation Logic (keep as is)
    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        return reply
    else:
        # Note: If 'rerun' also calls the AI, it will need a similar fix inside it!
        return rerun(reply, message, history, evaluation.feedback)

In [150]:
def chat(message, history):
    messages = []
    
    # Updated loop to handle Gradio's message dictionary format
    for entry in history:
        messages.append({"role": "user", "content": entry["human"]})
        messages.append({"role": "assistant", "content": entry["AI"]})

    # Add the current question
    messages.append({"role": "user", "content": message})

    try:
        response = ollama.chat(model='llama3.2', messages=messages)
        return response['message']['content']
        
    except Exception as e:
        return f"System Error: Ensure Ollama is running. ({str(e)})"

In [153]:
# Set type="messages" here
gr.ChatInterface(chat, type="messages").launch()

TypeError: ChatInterface.__init__() got an unexpected keyword argument 'type'

In [144]:

# Let's just check emails are working for you

def send_test_email():
    api_key = os.environ.get("SENDGRID_API_KEY")
    if not api_key:
        raise RuntimeError("SENDGRID_API_KEY is missing. Add it to your .env then rerun load_dotenv(override=True).")

    sg = SendGridAPIClient(api_key=api_key)
    from_email = Email("sajjachetansurya2799@gmail.com")  # must be a verified sender in SendGrid
    to_email = To("chetansurya2777@gmai.com")
    content = Content("text/plain", "This is an important test email")
    mail = Mail(from_email, to_email, "Test email", content).get()

    try:
        response = sg.client.mail.send.post(request_body=mail)
        print(response.status_code)
        if response.status_code != 202:
            print(getattr(response, "body", b""))
    except Exception as e:
        msg = str(e)
        print(msg)
        print("If you see 403 Forbidden: verify sender + regenerate API key with Mail Send permission.")
        raise